# Classical machine learning

Compare CSP+LDA on raw epochs against Logistic Regression, RBF-SVM, and Random Forest on band power. GroupKFold holds out full participants for the subject-independent estimate.

In [ ]:
from src.data import load_dataset
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay
from src.evaluation import evaluate_subject_independent, subject_independent_predictions

X, y, groups, channels, sfreq = load_dataset([1, 2, 3])
results = evaluate_subject_independent(
    X, y, groups, channels, sfreq, include_eegnet=False
)
display(results)
display(results.groupby('model')[['accuracy', 'balanced_accuracy', 'macro_f1', 'roc_auc']].agg(['mean', 'std']))

truth, predicted, scores = subject_independent_predictions(
    X, y, groups, channels, sfreq, model_name='csp_lda'
)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_predictions(
    truth, predicted, display_labels=['Left hand', 'Right hand'], ax=axes[0]
)
RocCurveDisplay.from_predictions(truth, scores, ax=axes[1], name='CSP + LDA')
plt.tight_layout()
from pathlib import Path
Path('results/figures').mkdir(parents=True, exist_ok=True)
fig.savefig('results/figures/subject_independent_csp.png', dpi=150, bbox_inches='tight')

summary = results.groupby('model')[['accuracy', 'balanced_accuracy', 'macro_f1', 'roc_auc']].mean()
summary.plot(kind='bar', ylim=(0, 1), figsize=(10, 5), rot=20)
plt.tight_layout()
plt.savefig('results/figures/model_comparison.png', dpi=150, bbox_inches='tight')